# Equity Options Pricing

**Comprehensive guide to pricing equity options with BSM, Monte Carlo, and Finite Difference methods**

This notebook demonstrates equity option pricing including:
- European and American vanilla options
- Dividend impact analysis
- Pricer comparison (BSM vs MC vs FDE)
- Greeks computation

---

## Table of Contents

1. [Setup](#1-setup)
2. [European Vanilla Options](#2-european-vanilla-options)
3. [American Options](#3-american-options)
4. [Dividend Impact](#4-dividend-impact)
5. [Pricer Comparison](#5-pricer-comparison)
6. [Greeks Analysis](#6-greeks-analysis)

In [ ]:
# Standard imports
import sys
sys.path.insert(0, '../../..')

import numpy as np
import matplotlib.pyplot as plt

# QuantStrata imports
from src.marketdata.core.ids import MarketId
from src.marketdata.core.market import Market
from src.marketdata.core.interfaces import Quote
from src.marketdata.surfaces.vol_surface import FlatVolSurface
from src.marketdata.curves.term_structure import FlatZeroRateCurve

# Instruments
from src.instruments.equity.options.vanilla import (
    EquityVanillaEuropeanOption, EquityVanillaAmericanOption,
)

# Pricers
from src.pricers.equity.european_bsm import EquityVanillaEuropeanOptionBsmPricer
from src.pricers.equity.european_bsm_mc import EquityVanillaEuropeanOptionMcPricer
from src.pricers.equity.american_bsm_fde import EquityVanillaAmericanOptionFdPricer

# Plot configuration
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 11,
})

print("Imports successful!")

---

## 1. Setup

### Define Market Identifiers

In [ ]:
# Define standard MarketIds for equity pricing
# These are used consistently across all examples

SPOT_ID = MarketId("EQUITY", "SPOT", "STOCK")
VOL_ID = MarketId("EQUITY", "VOL", "STOCK")
CURVE_ID = MarketId("IR", "CURVE", "USD.OIS", (("ccy", "USD"),))

print(f"Spot ID:  {SPOT_ID.key()}")
print(f"Vol ID:   {VOL_ID.key()}")
print(f"Curve ID: {CURVE_ID.key()}")

In [ ]:
def create_equity_market(
    spot: float = 100.0,
    vol: float = 0.20,
    rate: float = 0.05,
) -> Market:
    """
    Create a simple equity market for pricing.
    
    Parameters
    ----------
    spot : float
        Stock price.
    vol : float
        Implied volatility.
    rate : float
        Risk-free rate (continuously compounded).
    
    Returns
    -------
    Market
        Configured equity market snapshot.
    """
    return Market(
        asof="2026-01-15",
        quotes={
            SPOT_ID: Quote(value=spot),
        },
        curves={
            CURVE_ID: FlatZeroRateCurve(continuously_compounded_rate=rate),
        },
        vols={
            VOL_ID: FlatVolSurface(sigma=vol),
        },
    )

# Create base market
market = create_equity_market(spot=100.0, vol=0.20, rate=0.05)

print("Market created:")
print(f"  Spot: ${market.quote(SPOT_ID):.2f}")
print(f"  Vol:  {market.vol_surface(VOL_ID).vol(1.0, 100.0):.0%}")
print(f"  Rate: {market.curve(CURVE_ID).zero_rate(1.0):.1%}")

In [ ]:
def create_call_option(strike: float, expiry: float, div_yield: float = 0.02) -> EquityVanillaEuropeanOption:
    """Helper to create a European call option."""
    return EquityVanillaEuropeanOption(
        ticker="STOCK",
        option_type="call",
        strike=strike,
        expiry=expiry,
        notional=1.0,
        dividend_yield=div_yield,
        spot_id=SPOT_ID,
        vol_id=VOL_ID,
        curve_id=CURVE_ID,
    )

def create_put_option(strike: float, expiry: float, div_yield: float = 0.02) -> EquityVanillaEuropeanOption:
    """Helper to create a European put option."""
    return EquityVanillaEuropeanOption(
        ticker="STOCK",
        option_type="put",
        strike=strike,
        expiry=expiry,
        notional=1.0,
        dividend_yield=div_yield,
        spot_id=SPOT_ID,
        vol_id=VOL_ID,
        curve_id=CURVE_ID,
    )

def create_american_call(strike: float, expiry: float, div_yield: float = 0.02) -> EquityVanillaAmericanOption:
    """Helper to create an American call option."""
    return EquityVanillaAmericanOption(
        ticker="STOCK",
        option_type="call",
        strike=strike,
        expiry=expiry,
        notional=1.0,
        dividend_yield=div_yield,
        spot_id=SPOT_ID,
        vol_id=VOL_ID,
        curve_id=CURVE_ID,
    )

def create_american_put(strike: float, expiry: float, div_yield: float = 0.02) -> EquityVanillaAmericanOption:
    """Helper to create an American put option."""
    return EquityVanillaAmericanOption(
        ticker="STOCK",
        option_type="put",
        strike=strike,
        expiry=expiry,
        notional=1.0,
        dividend_yield=div_yield,
        spot_id=SPOT_ID,
        vol_id=VOL_ID,
        curve_id=CURVE_ID,
    )

print("Option factories defined.")

---

## 2. European Vanilla Options

### Pricing with BSM

In [ ]:
# Create ATM options (K=100, T=1Y, q=2%)
call_option = create_call_option(strike=100.0, expiry=1.0, div_yield=0.02)
put_option = create_put_option(strike=100.0, expiry=1.0, div_yield=0.02)

# Create BSM pricer
bsm_pricer = EquityVanillaEuropeanOptionBsmPricer()

# Price
call_price = bsm_pricer.price(call_option, market)
put_price = bsm_pricer.price(put_option, market)

print(f"ATM Call (K=100, T=1Y, q=2%): ${call_price:.4f}")
print(f"ATM Put  (K=100, T=1Y, q=2%): ${put_price:.4f}")

# Verify put-call parity: C - P = S*e^(-qT) - K*e^(-rT)
S, K, T, r, q = 100.0, 100.0, 1.0, 0.05, 0.02
parity_lhs = call_price - put_price
parity_rhs = S * np.exp(-q * T) - K * np.exp(-r * T)
print(f"\nPut-Call Parity Check:")
print(f"  C - P = {parity_lhs:.4f}")
print(f"  S*e^(-qT) - K*e^(-rT) = {parity_rhs:.4f}")
print(f"  Match: {np.isclose(parity_lhs, parity_rhs)}")

In [ ]:
# Price across strikes
strikes = np.linspace(70, 130, 25)
call_prices = []
put_prices = []

for K in strikes:
    call = create_call_option(strike=K, expiry=1.0)
    put = create_put_option(strike=K, expiry=1.0)
    call_prices.append(bsm_pricer.price(call, market))
    put_prices.append(bsm_pricer.price(put, market))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(strikes, call_prices, 'b-', label='Call')
ax.plot(strikes, put_prices, 'r-', label='Put')
ax.axvline(100, color='gray', linestyle='--', alpha=0.5, label='ATM')
ax.set_xlabel('Strike')
ax.set_ylabel('Option Price ($)')
ax.set_title('European Option Prices vs Strike')
ax.legend()

# Intrinsic value comparison
ax = axes[1]
spot = 100.0
call_intrinsic = np.maximum(spot - strikes, 0)
put_intrinsic = np.maximum(strikes - spot, 0)
call_time_value = np.array(call_prices) - call_intrinsic
put_time_value = np.array(put_prices) - put_intrinsic

ax.plot(strikes, call_time_value, 'b-', label='Call Time Value')
ax.plot(strikes, put_time_value, 'r-', label='Put Time Value')
ax.axvline(100, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Strike')
ax.set_ylabel('Time Value ($)')
ax.set_title('Time Value vs Strike')
ax.legend()

plt.tight_layout()
plt.show()

---

## 3. American Options

### Early Exercise Premium

In [ ]:
# American options (K=100, T=1Y, q=2%)
american_call = create_american_call(strike=100.0, expiry=1.0, div_yield=0.02)
american_put = create_american_put(strike=100.0, expiry=1.0, div_yield=0.02)

# FDE pricer for American options
fde_pricer = EquityVanillaAmericanOptionFdPricer()

american_call_price = fde_pricer.price(american_call, market)
american_put_price = fde_pricer.price(american_put, market)

print(f"American Call (K=100, T=1Y): ${american_call_price:.4f}")
print(f"American Put  (K=100, T=1Y): ${american_put_price:.4f}")

print(f"\nEarly Exercise Premium:")
print(f"  Call: ${american_call_price - call_price:.4f}")
print(f"  Put:  ${american_put_price - put_price:.4f}")

In [ ]:
# Compare European vs American across strikes
strikes = np.linspace(70, 130, 21)
eur_puts = []
amer_puts = []

for K in strikes:
    eur = create_put_option(strike=K, expiry=1.0)
    amer = create_american_put(strike=K, expiry=1.0)
    eur_puts.append(bsm_pricer.price(eur, market))
    amer_puts.append(fde_pricer.price(amer, market))

early_exercise = np.array(amer_puts) - np.array(eur_puts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(strikes, eur_puts, 'b-', label='European Put')
ax.plot(strikes, amer_puts, 'r--', label='American Put')
ax.axvline(100, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Strike')
ax.set_ylabel('Put Price ($)')
ax.set_title('European vs American Put')
ax.legend()

ax = axes[1]
ax.plot(strikes, early_exercise, 'g-', linewidth=2)
ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax.axvline(100, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Strike')
ax.set_ylabel('Early Exercise Premium ($)')
ax.set_title('Early Exercise Premium (American - European)')

plt.tight_layout()
plt.show()

print(f"Max early exercise premium: ${max(early_exercise):.4f} at K={strikes[np.argmax(early_exercise)]:.0f}")

---

## 4. Dividend Impact

### Effect of Dividend Yield

In [ ]:
# Price options with different dividend yields
div_yields = np.linspace(0, 0.10, 21)  # 0% to 10%

call_prices_by_div = []
put_prices_by_div = []

for q in div_yields:
    call = create_call_option(strike=100.0, expiry=1.0, div_yield=q)
    put = create_put_option(strike=100.0, expiry=1.0, div_yield=q)
    call_prices_by_div.append(bsm_pricer.price(call, market))
    put_prices_by_div.append(bsm_pricer.price(put, market))

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(div_yields * 100, call_prices_by_div, 'b-', label='ATM Call', linewidth=2)
ax.plot(div_yields * 100, put_prices_by_div, 'r-', label='ATM Put', linewidth=2)
ax.axvline(2, color='gray', linestyle='--', alpha=0.5, label='Base case (2%)')
ax.set_xlabel('Dividend Yield (%)')
ax.set_ylabel('Option Price ($)')
ax.set_title('Option Price vs Dividend Yield (K=100, T=1Y)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Intuition:")
print("  - Higher dividend → Lower forward → Lower call, Higher put")
print("  - Dividends reduce expected stock growth")

In [ ]:
# Forward price impact
S, r, T = 100.0, 0.05, 1.0

print("Forward price vs dividend yield:")
print(f"{'Div Yield':<12} {'Forward':<12} {'Change':<12}")
print("-" * 36)

base_forward = S * np.exp((r - 0.02) * T)
for q in [0, 0.02, 0.04, 0.06, 0.08, 0.10]:
    F = S * np.exp((r - q) * T)
    change = (F / base_forward - 1) * 100
    print(f"{q:.0%}{'':>8} ${F:.2f}{'':>5} {change:+.1f}%")

---

## 5. Pricer Comparison

### BSM vs Monte Carlo

In [ ]:
# Create MC pricer
mc_pricer = EquityVanillaEuropeanOptionMcPricer(n_paths=100000, seed=42)

# Compare prices
call_option = create_call_option(strike=100.0, expiry=1.0)

bsm_price = bsm_pricer.price(call_option, market)
mc_result = mc_pricer.price(call_option, market)

# MC returns simulation result with price attribute
mc_price = mc_result.price if hasattr(mc_result, 'price') else mc_result

print(f"ATM Call (K=100, T=1Y):")
print(f"  BSM:          ${bsm_price:.4f}")
print(f"  Monte Carlo:  ${mc_price:.4f}")
print(f"  Difference:   ${abs(bsm_price - mc_price):.4f} ({abs(bsm_price - mc_price)/bsm_price:.2%})")

In [ ]:
# Monte Carlo convergence
path_counts = [100, 500, 1000, 5000, 10000, 50000, 100000]
mc_prices = []
mc_errors = []

for n in path_counts:
    mc = EquityVanillaEuropeanOptionMcPricer(n_paths=n, seed=42)
    result = mc.price(call_option, market)
    price = result.price if hasattr(result, 'price') else result
    mc_prices.append(price)
    mc_errors.append(abs(price - bsm_price))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.semilogx(path_counts, mc_prices, 'bo-', markersize=8)
ax.axhline(bsm_price, color='red', linestyle='--', label=f'BSM: ${bsm_price:.4f}')
ax.set_xlabel('Number of Paths')
ax.set_ylabel('Option Price ($)')
ax.set_title('Monte Carlo Convergence')
ax.legend()

ax = axes[1]
ax.loglog(path_counts, mc_errors, 'go-', markersize=8)
# Expected O(1/sqrt(N)) convergence
x = np.array(path_counts)
if mc_errors[0] > 0:
    ax.loglog(x, mc_errors[0] * np.sqrt(path_counts[0]) / np.sqrt(x), 'r--', label=r'$O(1/\sqrt{N})$')
ax.set_xlabel('Number of Paths')
ax.set_ylabel('Absolute Error ($)')
ax.set_title('Monte Carlo Error Convergence')
ax.legend()

plt.tight_layout()
plt.show()

---

## 6. Greeks Analysis

### Computing Greeks

In [ ]:
# Get Greeks from BSM pricer
call_option = create_call_option(strike=100.0, expiry=1.0)
put_option = create_put_option(strike=100.0, expiry=1.0)

call_greeks = bsm_pricer.greeks(call_option, market)
put_greeks = bsm_pricer.greeks(put_option, market)

print("Greeks for ATM Options (K=100, T=1Y):")
print(f"\n{'Greek':<12} {'Call':<15} {'Put':<15}")
print("-" * 42)
for greek in ['delta', 'gamma', 'vega', 'theta', 'rho']:
    if greek in call_greeks:
        call_val = call_greeks[greek]
        put_val = put_greeks[greek]
        print(f"{greek.capitalize():<12} {call_val:<15.6f} {put_val:<15.6f}")

In [ ]:
# Delta across strikes
strikes = np.linspace(70, 130, 61)
call_deltas = []
put_deltas = []
gammas = []

for K in strikes:
    call = create_call_option(strike=K, expiry=1.0)
    put = create_put_option(strike=K, expiry=1.0)
    call_greeks = bsm_pricer.greeks(call, market)
    put_greeks = bsm_pricer.greeks(put, market)
    call_deltas.append(call_greeks['delta'])
    put_deltas.append(put_greeks['delta'])
    gammas.append(call_greeks['gamma'])  # Same for call and put

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(strikes, call_deltas, 'b-', label='Call Delta', linewidth=2)
ax.plot(strikes, put_deltas, 'r-', label='Put Delta', linewidth=2)
ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.axhline(-0.5, color='gray', linestyle='--', alpha=0.5)
ax.axvline(100, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Strike')
ax.set_ylabel('Delta')
ax.set_title('Delta vs Strike')
ax.legend()

ax = axes[1]
ax.plot(strikes, gammas, 'g-', linewidth=2)
ax.axvline(100, color='gray', linestyle='--', alpha=0.5, label='ATM')
ax.set_xlabel('Strike')
ax.set_ylabel('Gamma')
ax.set_title('Gamma vs Strike')
ax.legend()

plt.tight_layout()
plt.show()

print(f"Maximum Gamma: {max(gammas):.6f} at K={strikes[np.argmax(gammas)]:.1f}")

In [ ]:
# Vega and Theta across expiries
expiries = np.linspace(0.1, 2.0, 20)
vegas = []
thetas = []

for T in expiries:
    call = create_call_option(strike=100.0, expiry=T)
    greeks = bsm_pricer.greeks(call, market)
    vegas.append(greeks['vega'])
    thetas.append(greeks['theta'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(expiries, vegas, 'purple', linewidth=2)
ax.set_xlabel('Time to Expiry (years)')
ax.set_ylabel('Vega')
ax.set_title('ATM Vega vs Expiry')

ax = axes[1]
ax.plot(expiries, thetas, 'orange', linewidth=2)
ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Time to Expiry (years)')
ax.set_ylabel('Theta (per year)')
ax.set_title('ATM Theta vs Expiry')

plt.tight_layout()
plt.show()

print("Observations:")
print("  - Vega increases with √T (more time = more vol sensitivity)")
print("  - Theta is most negative for short-dated ATM options")

---

## Summary

**Key Takeaways:**

1. **BSM** is exact for European options with continuous dividends
2. **Monte Carlo** converges at $O(1/\sqrt{N})$ but handles path-dependence
3. **FDE** is essential for American options (early exercise)
4. **Dividends** reduce call value, increase put value
5. **Early exercise premium** is most significant for ITM puts
6. **Gamma** peaks at ATM, **Theta** is most negative at ATM